# 004 - Stage 4: generator likelihood

Stages 1–2 separate the families and outliers (val family accuracy 96%), but not the sizes: size accuracy given the correct family is at chance (25.3%). Stage 3 showed that marginal token statistics (histograms, entropy, repeats, codebook rarity) do not identify the size either, even from the true tokens.

**Hypothesis.** Within a family the variants share the autoencoder and differ only in their transformer. Each transformer should give the highest likelihood, under its **own guided sampling distribution**, to the tokens it generated. For an image with tokens $t_1 \dots t_L$ and class $c$, the per-model score
$$s_m = \sum_{n=1}^{L} \log \tilde p_m(t_n \mid t_{<n}, c)$$
should therefore identify the variant $m$.

**Guided sampling distributions**, rebuilt exactly as in each repo's sampler (`tracer/likelihood.py`), from the conditional logits $l_c$ (class $c$) and the unconditional logits $l_u$ (null class) at each position:
- **RAR** (`RAR.generate`), raster order, $n = 0 \dots 255$:
$$\tilde p = \mathrm{softmax}\Big(\big(l_u + (l_c - l_u)\, g_n\big) / T\Big), \qquad g_n = (w - 1)\,\frac{1 - \cos\!\big(\pi (n/256)^{\text{pow}}\big)}{2} + 1,$$
with $(w, \text{pow}, T)$ per size from `tracer.generate.RAR_SAMPLING`.
- **VAR** (`VAR.autoregressive_infer_cfg`), every token of scale $s = 0 \dots 9$:
$$\tilde p = \mathrm{softmax}\big((1 + \tau_s)\, l_c - \tau_s\, l_u\big), \qquad \tau_s = 1.5\, s / 9.$$
The sampler then keeps only the top-$k$ = 900 / top-$p$ = 0.96 codes. $\log \tilde p$ is computed **without** this filter, so it stays finite. Whether the token survives it is recorded separately (`in_support`).

**Per-token quantities** (`scripts/score_likelihood.py`): `logp_guided` $= \log \tilde p_m(t_n)$; `logp_cond` and `logp_uncond`, the log-softmax of $l_c$ and $l_u$ at $t_n$; `rank_guided`, the number of codes with a larger guided logit; and for VAR `in_support`. Per image they are summarised by their sum and mean, `min20_logp_guided` (mean of the lowest 20% of `logp_guided`), `mean_cond_minus_uncond` and, for VAR, `in_support_rate`.

**Phases.** A oracle (true tokens, true class), B1 (recovered tokens, true class), B2 realistic (recovered tokens, estimated class), C transfer (task images), D end to end.

**Pre-registered criteria** (per family):
| phase | data | pass if |
|---|---|---|
| A oracle | true tokens, true class, generated holdout | 4-way accuracy $\ge$ 40% |
| B2 realistic | recovered tokens, estimated class, generated holdout | adjacent-size AUC $\ge$ 0.60 |
| C transfer | task images | evaluated only if B2 holdout accuracy $\ge$ 35%; then task accuracy within 10 points of holdout |

**Data and rules.** The fresh generated set `stage3/gen` (2,560 images per model, seed 1) with the same train / holdout split as Stage 3 (`stage3/gen_split.csv`, 80/20 stratified by model). Scores are in `stage4/scores/<family>/<source>_<tokens>_<class-mode>/<model>.npz`. Classifiers are trained on generated train only; RAR and VAR are analysed separately. This notebook only reads cached outputs and runs on the CPU.

Sections: **1. Data and score checks**; **2. Oracle (phase A)**; **3. Recovered tokens (B1) and estimated classes (B2)**; **4. Transfer to task images (phase C)**; **5. End to end (phase D)**; **6. Findings**.

## Setup

In [ ]:
import sys, json
from pathlib import Path
REPO = Path("/workspace/iar-model-tracer")
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from scipy.special import logsumexp, softmax

from tracer.common import CACHE, LABELS, build_index
from tracer.generate import MODELS
from tracer.metrics import confusion, family_confusion, report
from tracer.stage1 import KNOWN_FAMILIES, NearestFamilyThreshold
from tracer.tokens import SEQ_LEN

S3, S4 = CACHE / "stage3", CACHE / "stage4"
SCORES, TESTS = S4 / "scores", S4 / "tests"
SEED = 0        # bootstrap
N_BOOT = 2000   # bootstrap resamples for accuracy intervals (1,000 for pairwise AUCs)
ORACLE_RUN = "gen_true_true"  # source gen, true tokens, true class

# Per-image summaries in the score files ([N,K] with K = 1 classes here).
SUMMARIES = {"rar": ["sum_logp_cond", "mean_logp_cond", "sum_logp_uncond", "mean_logp_uncond", "sum_logp_guided",
                     "mean_logp_guided", "sum_rank_guided", "mean_rank_guided", "min20_logp_guided",
                     "mean_cond_minus_uncond"]}
SUMMARIES["var"] = SUMMARIES["rar"] + ["sum_in_support", "mean_in_support", "in_support_rate"]

## 1. Data and score checks

Checks below:
1. **Provenance** of each oracle score file (`gen_true_true/<model>.npz`): its `config.json` says generated source, true tokens, true class and no `--limit`. Rows match the stage3 token file (label, index, class) one to one, and the scored class is the true class. Values are finite and log-probabilities are $\le 0$. The float32 summaries agree with the float16 per-token arrays, and `sum` $= L \cdot$ `mean`.
2. **Consistency tests** of `tracer/likelihood.py` (`python -m tracer.test_likelihood`, cached in `stage4/tests/`): teacher-forced scores equal the samplers' own step-by-step distributions, and own-model top-$k$ / top-$p$ support is close to 1.
3. **Generating × scoring model tables** on generated train: mean `logp_guided`, mean `logp_cond - logp_uncond` and (VAR) the support rate, plus how often each scoring model gives an image the highest $s_m$.

In [ ]:
# Check 1: provenance and integrity of the oracle score files; build one table per family with the train / holdout
# split and one column per (scoring model, summary), e.g. "rarb:mean_logp_guided".
split = pd.read_csv(S3 / "gen_split.csv")
S, rows = {}, []
for fam in KNOWN_FAMILIES:
    with np.load(S3 / "tokens" / fam / "gen.npz") as z:
        tok = {k: z[k] for k in ("label", "index", "class")}
    sp = split[split.family == fam].set_index(["label", "index"]).split
    df = pd.DataFrame({"label": tok["label"], "index": tok["index"], "class": tok["class"]})
    df["split"] = sp.loc[list(zip(df.label, df["index"]))].values
    L = SEQ_LEN[fam]
    for m in MODELS[fam]:
        cfg = json.loads((SCORES / fam / ORACLE_RUN / m / "config.json").read_text())
        assert (cfg["family"], cfg["source"], cfg["tokens"], cfg["class_mode"], cfg["model"], cfg["limit"]) == \
            (fam, "gen", "true", "true", m, None), cfg
        with np.load(SCORES / fam / ORACLE_RUN / f"{m}.npz") as z:
            d = {k: z[k] for k in z.files}
        assert (d["row"] == np.arange(len(df))).all()
        assert (d["label"] == df.label.values).all() and (d["index"] == df["index"].values).all()
        assert (d["class"] == df["class"].values).all() and (d["classes"][:, 0] == d["class"]).all()
        assert (d["class_prob"] == 1).all() and d["logp_guided"].shape == (len(df), 1, L)
        lp = {k: d[k].astype(np.float32) for k in ("logp_cond", "logp_uncond", "logp_guided")}
        assert all(np.isfinite(v).all() and (v <= 0).all() for v in lp.values())
        assert all(np.isfinite(d[k]).all() for k in SUMMARIES[fam])
        rows.append({"family": fam, "model": m, "images": len(df), "tokens / image": L,
                     "max |mean(f16 per-token) - mean_logp_guided|": np.abs(lp["logp_guided"].mean(-1)
                                                                          - d["mean_logp_guided"]).max(),
                     "max |sum - L*mean| (logp_guided)": np.abs(d["sum_logp_guided"] - L * d["mean_logp_guided"]).max(),
                     "min20 <= mean": bool((d["min20_logp_guided"] <= d["mean_logp_guided"] + 1e-6).all()),
                     "logp_guided min": lp["logp_guided"].min()})
        for k in SUMMARIES[fam]:
            df[f"{m}:{k}"] = d[k][:, 0]
        del d, lp
    S[fam] = df
    print(fam, "split:", df.groupby(["label", "split"]).size().unstack().to_dict())
display(pd.DataFrame(rows).set_index(["family", "model"]))

### Consistency tests of `tracer/likelihood.py`

**Check 1, teacher forcing equals the samplers** (4 generated-train samples of RAR-L and VAR-d16). The teacher-forced guided logits are compared with the logits or probabilities the repos' samplers compute when forced to emit the same tokens:
- RAR: (a) a step-by-step loop without kv cache written like `RAR.generate`, and (b) `RAR.generate` itself, both fp32;
- VAR: `autoregressive_infer_cfg` itself, which runs under fp16 autocast; scored in fp32 (as in this notebook's scores) and fp16. The VAR differences are fp16 rounding in the sampler.

**Check 2, own-model support** (VAR, 64 generated-train samples per depth): the fraction of tokens inside the scoring model's top-$k$ / top-$p$ support. It should be about 1 when the scoring model generated the tokens.

In [ ]:
tests = json.loads((TESTS / "test_likelihood.json").read_text())
print("run:", tests["meta"])
keep = {"rar": ["label", "logits", "|guided logit| max", "(a) max |dlogit| step-by-step", "(a) max |dlogp(t)| step-by-step",
                "(b) max |dprob| generate()", "(b) max |dlogp(t)| generate()"],
        "var": ["label", "logits", "|guided logit| max", "own top-k/top-p mask == repo mask (all codes)",
                "sampler in_support at t", "fp32: max |dlogit|", "fp32: mean |dlogit|", "fp32: max |dlogp(t)|",
                "fp32: in_support agrees with sampler", "fp16: max |dlogit|", "fp16: max |dlogp(t)|"]}
for fam in KNOWN_FAMILIES:
    display(pd.Series({k: tests[fam][k] for k in keep[fam]}, name=f"check 1, {fam}").to_frame())

sup = pd.read_csv(TESTS / "test_likelihood_support.csv")
display(sup.pivot(index="scoring model", columns="generated by", values="in_support")
        .style.format("{:.4f}").set_caption("check 2: VAR in_support rate (rows = scoring model)"))
display(sup[sup["scoring model"] == sup["generated by"]].set_index("scoring model")
        [["in_support", "in_support (fp16)", "images fully in support"]].style.format("{:.4f}"))

### Generating × scoring model, on generated train

Rows are the model that generated the images (2,048 train images each), columns the model that scores them. Each cell is the mean over images of a per-image summary. If the hypothesis holds, each row should be most favourable on its diagonal (highlighted: the best cell of each row). Raw values are not comparable across columns: each scoring model has its own guidance and temperature, so its likelihoods have their own scale. The rule in Section 2 accounts for this by standardising each scoring model on its own samples.

The last table is the share of each row's images for which the scoring model gives the highest $s_m = \sum_n \log \tilde p_m$ (raw argmax, no standardisation; rows sum to 1).

In [ ]:
def gen_x_score(df, fam, summary):
    tr = df[df.split == "train"]
    return pd.DataFrame({m: tr.groupby("label")[f"{m}:{summary}"].mean() for m in MODELS[fam]}).loc[MODELS[fam]] \
        .rename_axis(index="generated by", columns="scored by")


for fam in KNOWN_FAMILIES:
    df = S[fam]
    tables = [("mean_logp_guided", "mean logp_guided per token", "max"),
              ("mean_cond_minus_uncond", "mean logp_cond - logp_uncond per token", "max")]
    if fam == "var":
        tables.append(("in_support_rate", "in_support rate (top-k 900 / top-p 0.96)", "max"))
    for summary, title, best in tables:
        t = gen_x_score(df, fam, summary)
        display(t.style.format("{:.4f}").highlight_max(axis=1, color="#fff3c4").set_caption(f"{fam}: {title}"))
    tr = df[df.split == "train"]
    s = tr[[f"{m}:sum_logp_guided" for m in MODELS[fam]]].values
    best = np.array(MODELS[fam])[s.argmax(1)]
    share = pd.crosstab(tr.label, best, normalize="index").reindex(index=MODELS[fam], columns=MODELS[fam], fill_value=0)
    display(share.rename_axis(index="generated by", columns="highest s_m").style.format("{:.3f}")
            .highlight_max(axis=1, color="#fff3c4").set_caption(f"{fam}: share of images with the highest s_m"))

## 2. Oracle (phase A): true tokens, true class

**Question.** Given the generator's own tokens and the true class, which model wrote them? If the sizes cannot be separated here, recovered tokens and estimated classes will not help.

**(a) Rule: best fit to each model's own samples.** For each scoring model $m$, the score of image $x$ is $s_m(x) = \sum_n \log \tilde p_m(t_n \mid t_{<n}, c)$. Its typical value on the model's **own** samples is estimated on generated train:
$$\mu_m = \operatorname{mean}_{x \in \mathcal T_m} s_m(x), \qquad \sigma_m = \operatorname{sd}_{x \in \mathcal T_m} s_m(x), \qquad z_m(x) = \frac{s_m(x) - \mu_m}{\sigma_m},$$
with $\mathcal T_m$ the train images generated by $m$. The prediction is the model whose own-sample distribution the image fits best, $\hat m = \arg\min_m |z_m(x)|$, and $-|z_m(x)|$ is the score of class $m$ for the pairwise AUCs. Standardising removes each model's own scale (guidance, temperature, model quality), which raw $\arg\max_m s_m$ ignores. The two-sided $|z|$ treats a likelihood that is unusually high for $m$ as just as unlike $m$'s samples as one that is unusually low. Fixed in advance: the score is $s_m$ (sum of `logp_guided`), the fit is $|z|$, and nothing is tuned.

**(b) Logistic regression on all per-model summaries.** For each of the 4 scoring models: mean `logp_cond`, `logp_uncond`, `logp_guided` and `rank_guided`, `min20_logp_guided`, and for VAR `in_support_rate`. That gives 20 features for RAR and 24 for VAR. The other summaries in the files are exact linear copies of these: `sum_q` $= L \cdot$ `mean_q` (L is fixed), `mean_cond_minus_uncond` $=$ mean `logp_cond` $-$ mean `logp_uncond`, and `sum` / `mean_in_support` $\propto$ `in_support_rate`. They are left out because they add no information and would only change how the L2 penalty is shared. Pipeline as in Stage 3: `StandardScaler` → multinomial `LogisticRegression` ($C = 1$).

**Evaluation** as in 003, on the generated holdout (512 images per size): 4-way accuracy with a 95% percentile bootstrap interval (2,000 resamples), confusion matrix (rows = true), and pairwise AUC for every pair $(a, b)$, $a$ smaller, from the class-score difference $\mathrm{score}_b - \mathrm{score}_a$ on the holdout images of $a$ and $b$ (log-probabilities for (b), $-|z|$ for (a)). Adjacent pairs and the extreme pair are highlighted.

In [ ]:
def boot_ci(stat, n, rng, n_boot=N_BOOT):
    # 95% percentile interval of stat(idx) over bootstrap resamples idx of range(n).
    vals = [stat(rng.integers(0, n, n)) for _ in range(n_boot)]
    return np.percentile(vals, [2.5, 97.5])


def evaluate(scores, y, sizes):
    # scores: [N,4] DataFrame, one column per size, higher = more likely. Accuracy (+ CI), confusion, pairwise AUCs.
    rng = np.random.default_rng(SEED)
    scores = scores[sizes]
    pred = np.array(sizes)[scores.values.argmax(1)]
    correct = pred == y
    acc_lo, acc_hi = boot_ci(lambda b: correct[b].mean(), len(y), rng)
    cm = pd.DataFrame(confusion_matrix(y, pred, labels=sizes), index=pd.Index(sizes, name="true"),
                      columns=pd.Index(sizes, name="predicted"))
    pairs = []
    for i, a in enumerate(sizes):
        for b in sizes[i + 1:]:
            m = np.isin(y, [a, b])
            yb, s = (y[m] == b), (scores[b] - scores[a]).values[m]
            lo, hi = boot_ci(lambda r: roc_auc_score(yb[r], s[r]), m.sum(), rng, n_boot=1000)
            gap = sizes.index(b) - i
            kind = "adjacent" if gap == 1 else "extreme" if gap == len(sizes) - 1 else ""
            pairs.append({"pair": f"{a} vs {b}", "type": kind, "AUC": roc_auc_score(yb, s), "lo": lo, "hi": hi})
    return {"acc": correct.mean(), "acc_lo": acc_lo, "acc_hi": acc_hi, "cm": cm, "pairs": pd.DataFrame(pairs)}


class OwnSampleRule:
    # (a): z-score s_m with mean / sd of model m's own train samples; predict argmin_m |z_m|.
    def __init__(self, sizes, col="sum_logp_guided"):
        self.sizes, self.col = sizes, col

    def fit(self, df):
        own = {m: df.loc[df.label == m, f"{m}:{self.col}"] for m in self.sizes}
        self.mu_ = {m: v.mean() for m, v in own.items()}
        self.sd_ = {m: v.std() for m, v in own.items()}
        return self

    def z(self, df):
        return pd.DataFrame({m: (df[f"{m}:{self.col}"] - self.mu_[m]) / self.sd_[m] for m in self.sizes}, index=df.index)

    def scores(self, df):
        return -self.z(df).abs()


def lr_features(fam):
    per_model = ["mean_logp_cond", "mean_logp_uncond", "mean_logp_guided", "mean_rank_guided", "min20_logp_guided"]
    per_model += ["in_support_rate"] if fam == "var" else []
    return [f"{m}:{k}" for m in MODELS[fam] for k in per_model]


oracle = {}
for fam in KNOWN_FAMILIES:
    df, sizes = S[fam], MODELS[fam]
    tr, ho = df[df.split == "train"], df[df.split == "holdout"]
    rule = OwnSampleRule(sizes).fit(tr)
    feats = lr_features(fam)
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(tr[feats], tr.label)
    lr_scores = lambda d: pd.DataFrame(lr.predict_log_proba(d[feats]), columns=lr.classes_, index=d.index)
    oracle[fam] = {
        "rule": {**evaluate(rule.scores(ho), ho.label.values, sizes),
                 "train_acc": (np.array(sizes)[rule.scores(tr).values.argmax(1)] == tr.label.values).mean()},
        "lr": {**evaluate(lr_scores(ho), ho.label.values, sizes), "train_acc": lr.score(tr[feats], tr.label)},
        "rule_model": rule, "lr_model": lr, "features": feats,
    }
    print(f"{fam}: rule mu / sd of own s_m:", {m: f"{rule.mu_[m]:.1f} / {rule.sd_[m]:.1f}" for m in sizes},
          f"| LR features: {len(feats)}")

### Holdout accuracy and confusion matrices

In [ ]:
CLFS = {"rule": "(a) own-sample rule", "lr": "(b) logistic regression"}
rows = []
for fam in KNOWN_FAMILIES:
    for clf, name in CLFS.items():
        r = oracle[fam][clf]
        rows.append({"family": fam, "classifier": name, "train acc (in-sample)": r["train_acc"], "holdout acc": r["acc"],
                     "95% CI": f"[{r['acc_lo']:.3f}, {r['acc_hi']:.3f}]"})
oracle_acc = pd.DataFrame(rows).set_index(["family", "classifier"])
display(oracle_acc.style.format({"train acc (in-sample)": "{:.3f}", "holdout acc": "{:.3f}"}))

for fam in KNOWN_FAMILIES:
    display(pd.concat({f"{fam} {name}": oracle[fam][clf]["cm"] for clf, name in CLFS.items()}, axis=1)
            .style.set_caption(f"{fam}: holdout confusion (rows = true, 512 per size)"))

### Pairwise AUC

Adjacent pairs are shaded yellow, the extreme pair blue. 95% bootstrap intervals (1,000 resamples).

In [ ]:
def shade(row):
    colour = {"adjacent": "background-color: #fff3c4", "extreme": "background-color: #d6e9ff"}.get(row["type"], "")
    return [colour] * len(row)


for fam in KNOWN_FAMILIES:
    parts = {}
    for clf, name in CLFS.items():
        p = oracle[fam][clf]["pairs"].set_index(["pair", "type"])
        parts[clf] = p.assign(CI=[f"[{a:.3f}, {b:.3f}]" for a, b in zip(p.lo, p.hi)])[["AUC", "CI"]]
    t = pd.concat(parts, axis=1).reset_index()
    t.columns = [" ".join(c).strip() if isinstance(c, tuple) else c for c in t.columns]
    display(t.style.apply(shade, axis=1).format({c: "{:.3f}" for c in t.columns if c.endswith("AUC")})
            .set_caption(f"{fam}: holdout AUC per pair of sizes (rule = (a), lr = (b))").hide(axis="index"))

### The rule's z-scores

For each scoring model $m$ (one panel), the distribution of $z_m$ on the holdout images of each generating model. By construction the model's own images (bold) are centred near 0 with spread about 1; the rule works to the extent that the other models' images sit away from 0.

In [ ]:
for fam in KNOWN_FAMILIES:
    df, sizes = S[fam], MODELS[fam]
    ho = df[df.split == "holdout"]
    z = oracle[fam]["rule_model"].z(ho)
    fig, axs = plt.subplots(1, 4, figsize=(18, 3.4), sharey=True)
    for ax, m in zip(axs, sizes):
        lo, hi = np.percentile(z[m], [0.5, 99.5])
        bins = np.linspace(lo, hi, 60)
        for g in sizes:
            ax.hist(z.loc[ho.label == g, m], bins=bins, histtype="step", density=True, label=g, lw=2.2 if g == m else 1)
        ax.axvline(0, c="k", lw=0.6)
        ax.set_title(f"z under {m}"); ax.set_xlabel("z")
    axs[0].legend(title="generated by"); fig.suptitle(f"{fam}: holdout z-scores of s_m", y=1.02)
    plt.tight_layout(); plt.show()

### Diagnostic: paired comparison within an image

Not one of the two pre-specified classifiers; shown to explain the gap between (a) and (b). Raw $\arg\max_m s_m(x)$ compares the four models on the **same** image, with nothing fitted. The margin is $s_{\text{own}}(x) - \max_{m \ne \text{own}} s_m(x)$ in nats (total, and per token).

In [ ]:
rows = []
for fam in KNOWN_FAMILIES:
    df, sizes = S[fam], MODELS[fam]
    s = df[[f"{m}:sum_logp_guided" for m in sizes]].values
    own = np.array([sizes.index(l) for l in df.label])
    other = s.copy(); other[np.arange(len(s)), own] = -np.inf
    margin = s[np.arange(len(s)), own] - other.max(1)
    sd_own = np.mean([oracle[fam]["rule_model"].sd_[m] for m in sizes])
    for part in ("train", "holdout"):
        m = (df.split == part).values
        rows.append({"family": fam, "split": part, "raw argmax acc": np.mean(s[m].argmax(1) == own[m]),
                     "margin min": margin[m].min(), "margin p1": np.percentile(margin[m], 1),
                     "margin median": np.median(margin[m]), "median per token": np.median(margin[m]) / SEQ_LEN[fam],
                     "mean sd of own s_m (rule)": sd_own})
display(pd.DataFrame(rows).set_index(["family", "split"]).style.format("{:.3f}", subset=["raw argmax acc", "median per token"])
        .format("{:.1f}", subset=["margin min", "margin p1", "margin median", "mean sd of own s_m (rule)"]))

### Oracle criterion

Pre-registered: 4-way holdout accuracy $\ge$ 40% on true tokens with the true class, per family. "Pass" uses the point estimate; the lower end of the 95% interval is shown too.

In [ ]:
crit = oracle_acc[["holdout acc", "95% CI"]].copy()
crit["CI low"] = [oracle[f][c]["acc_lo"] for f in KNOWN_FAMILIES for c in CLFS]
crit["criterion"] = ">= 0.40"
crit["pass"] = crit["holdout acc"] >= 0.40
crit["pass (CI low >= 0.40)"] = crit["CI low"] >= 0.40
display(crit.style.format({"holdout acc": "{:.3f}", "CI low": "{:.3f}"}))

### Section 2 result

**The oracle criterion passes for both families, by a wide margin.** With the true tokens and the true class, the generating model is identified almost perfectly:

| family | (a) own-sample rule | (b) logistic regression | criterion ($\ge$ 40%) |
|---|---|---|---|
| RAR | 98.6% [98.0, 99.1] | 100% [100, 100] | pass |
| VAR | 81.8% [80.2, 83.5] | 100% [100, 100] | pass |

1. **Each transformer strongly prefers its own tokens.** In every generating × scoring table the diagonal is the best cell of its row (mean `logp_guided` and `logp_cond − logp_uncond`; for VAR also the support rate). On every train and holdout image, the generating model gives the highest $s_m$. On the holdout it beats the best other model by at least 219 nats for RAR and 140 for VAR (train: 125 and 110), with medians of 626 and 397 nats (2.4 and 0.58 per token). This is what the hypothesis predicts, and the opposite of Stage 3: the information is in how probable each token is given its context under each model, not in the marginal token statistics.
2. **The rule (a) loses information by standardising each model separately.** An image's content shifts all four $s_m$ together, by more than the between-model gap when each model is viewed alone ($\sigma_m$ = 114 nats for RAR, 207–287 for VAR). Other models' images therefore often sit within $|z| \lesssim 1.5$ of a model's own samples, especially for VAR (z-score plots). The confusions are mostly towards VAR-d16 and RAR-B. The logistic regression sees all four scores of the same image and recovers the paired comparison, as does raw $\arg\max_m s_m$ (diagnostic: 100%).
3. **This is the easiest case.** The tokens are exact samples from the scored distribution, with the true class. The realistic setting replaces about 15% of RAR tokens and 73% of VAR tokens (Stage 3, Section 1) and uses an estimated class (top-1 correct for about 93% of generated images in the `estimate_classes.py` dry run). VAR's per-token margin (0.58 nats) is much smaller than RAR's (2.4), and far more of its tokens are replaced, so it is the family most at risk in phase B.

## 3. Recovered tokens (B1) and estimated classes (B2)

Phase A used the generator's own tokens and the true class. In practice there is only the image, so both have to be estimated:
- **B1, recovered tokens, true class.** The tokens are $\hat t = Q(D^{-1}(x))$, recovered with the Stage 2 $D^{-1}$ (`stage3/tokens/<family>/gen.npz`; about 85% of RAR and 27% of VAR tokens are recovered exactly). The class is still the true one. A → B1 is the loss from **token recovery** alone.
- **B2, recovered tokens, estimated class** (the realistic setting). The class comes from a pretrained ImageNet classifier (`scripts/estimate_classes.py`, convnext_large). Each image is scored with each of its top $K = 5$ classes $c_1 \dots c_5$, with classifier probabilities $\pi_1 \dots \pi_5$, renormalised over the five: $\tilde\pi_k = \pi_k / \sum_j \pi_j$. B1 → B2 is the loss from **not knowing the class**.

**Combining the $K$ class-conditional scores of model $m$** into one value per summary, with $s_{mk} = \sum_n \log \tilde p_m(\hat t_n \mid \hat t_{<n}, c_k)$:
- **(i) max over classes:** $k^*_m = \arg\max_k s_{mk}$, the class model $m$ finds most likely for these tokens. Every summary of model $m$ is taken at $k^*_m$, so $s_m = \max_k s_{mk}$.
- **(ii) mixture weighted by the classifier:** $s_m = \log \sum_k \tilde\pi_k \exp(s_{mk})$, the log-likelihood of the tokens under model $m$ with the class marginalised over the classifier's top-5. The other summaries (means, ranks, Min-20%, support) are averaged with the same mixture's class weights for model $m$, $w_{mk} = \tilde\pi_k e^{s_{mk}} / \sum_j \tilde\pi_j e^{s_{mj}}$.

`logp_uncond` uses the null class, so it is the same for every $k$ (checked below). Each model can pick a different class, so the $\arg\max$ in (i) and the weights in (ii) are per model.

**Classifiers and evaluation** exactly as in Section 2: the own-sample rule (a) and the logistic regression (b), **refit on each phase's own generated train** and evaluated on the same holdout images. Accuracies, confusions and pairwise AUCs are shown side by side for A, B1, B2 (i) and B2 (ii).

**Pre-registered realistic criterion:** adjacent-size AUC $\ge$ 0.60 on the generated holdout in B2. As in Stage 3, it is checked for every adjacent pair, and a classifier passes only if all three pairs do. Both combinations and both classifiers are reported. Phase C is evaluated only if B2 holdout accuracy is $\ge$ 35%.

In [ ]:
# Load a score run as per-model summary arrays [N,K], with provenance checks. Only summaries are read (the
# per-token arrays of the top-5 runs are large), plus logp_guided of one model for a float16 cross-check.
RUNS = {"A": "gen_true_true", "B1": "gen_recovered_true", "B2": "gen_recovered_topk"}
with np.load(S4 / "classes" / "gen.npz") as z:
    CLS = {k: z[k] for k in z.files}


def load_run(fam, run):
    raw, ref = {}, S[fam]
    for m in MODELS[fam]:
        cfg = json.loads((SCORES / fam / run / m / "config.json").read_text())
        src, tok, mode = run.split("_")
        assert (cfg["family"], cfg["source"], cfg["tokens"], cfg["class_mode"], cfg["model"], cfg["limit"]) == \
            (fam, src, tok, mode, m, None), cfg
        if mode == "topk":
            assert cfg["topk"] == 5 and cfg["classes"] is None, cfg  # default class file: stage4/classes/gen.npz
        z = np.load(SCORES / fam / run / f"{m}.npz")
        assert (z["row"] == np.arange(len(ref))).all() and (z["label"] == ref.label.values).all()
        assert (z["index"] == ref["index"].values).all() and (z["class"] == ref["class"].values).all()
        d = {k: z[k].astype(np.float32) for k in SUMMARIES[fam]}
        d["classes"], d["class_prob"] = z["classes"].astype(np.int64), z["class_prob"]
        if m == MODELS[fam][0]:
            lp = z["logp_guided"].astype(np.float32)
            assert np.isfinite(lp).all() and (lp <= 0).all()
            assert np.abs(lp.mean(-1) - d["mean_logp_guided"]).max() < 2e-3
            del lp
        assert all(np.isfinite(v).all() for k, v in d.items() if k in SUMMARIES[fam])
        raw[m] = d
    return raw


raw = {fam: {ph: load_run(fam, run) for ph, run in RUNS.items() if ph != "A"} for fam in KNOWN_FAMILIES}

# The B2 classes are the classifier's top-5, in the same rows as the class file.
rows = []
for fam in KNOWN_FAMILIES:
    ref = S[fam]
    fam_rows = np.flatnonzero(CLS["family"] == fam)
    assert (CLS["label"][fam_rows] == ref.label.values).all() and (CLS["index"][fam_rows] == ref["index"].values).all()
    assert (CLS["class"][fam_rows] == ref["class"].values).all()
    for m in MODELS[fam]:
        b2 = raw[fam]["B2"][m]
        assert (b2["classes"] == CLS["top10_class"][fam_rows, :5]).all()
        assert np.allclose(b2["class_prob"], CLS["top10_prob"][fam_rows, :5])
        # logp_uncond does not depend on the class (null-class logits): equal across the 5 slots up to float32 noise
        spread = np.abs(b2["mean_logp_uncond"] - b2["mean_logp_uncond"][:, :1]).max()
        assert spread < 1e-3, spread
    top = CLS["top10_class"][fam_rows]
    for g in MODELS[fam]:
        sel = (ref.label == g).values
        hit = top[sel] == ref["class"].values[sel, None]
        rows.append({"family": fam, "generated by": g, "classifier top-1": hit[:, 0].mean(), "top-5": hit[:, :5].any(1).mean(),
                     "top-5 prob mass (median)": np.median(CLS["top10_prob"][fam_rows][sel, :5].sum(1))})
display(pd.DataFrame(rows).set_index(["family", "generated by"]).style.format("{:.3f}")
        .set_caption("class estimates on all generated images (train + holdout)"))

In [ ]:
def combine(raw_run, fam, how):
    # Per-model summaries [N,K] -> one value per image: how = "single" (K = 1), "max" (i) or "mix" (ii).
    out = {}
    for m, d in raw_run.items():
        s = d["sum_logp_guided"]
        if how == "single":
            assert s.shape[1] == 1
            vals = {k: d[k][:, 0] for k in SUMMARIES[fam]}
        elif how == "max":
            k_star = s.argmax(1)
            vals = {k: d[k][np.arange(len(s)), k_star] for k in SUMMARIES[fam]}
            out[f"{m}:k_star"] = k_star
        else:
            log_pi = np.log(d["class_prob"] / d["class_prob"].sum(1, keepdims=True))
            a = log_pi + s.astype(np.float64)
            w = softmax(a, axis=1)
            vals = {k: (w * d[k]).sum(1) for k in SUMMARIES[fam]}
            vals["sum_logp_guided"] = logsumexp(a, axis=1)
            vals["mean_logp_guided"] = vals["sum_logp_guided"] / SEQ_LEN[fam]
            out[f"{m}:w_max"] = w.max(1)
        out.update({f"{m}:{k}": v for k, v in vals.items()})
    return pd.DataFrame(out)


def phase_table(fam, phase):
    base = S[fam][["label", "index", "class", "split"]]
    if phase == "A":
        return S[fam]
    if phase == "B1":
        return pd.concat([base, combine(raw[fam]["B1"], fam, "single")], axis=1)
    return pd.concat([base, combine(raw[fam]["B2"], fam, {"B2 (i) max": "max", "B2 (ii) mix": "mix"}[phase])], axis=1)


PHASES = ["A", "B1", "B2 (i) max", "B2 (ii) mix"]
T = {fam: {ph: phase_table(fam, ph) for ph in PHASES} for fam in KNOWN_FAMILIES}

# How the classes were used in B2: share of images where model m's best class (i) is the true class, and how
# concentrated the mixture weights (ii) are, for the generating model and for the others.
rows = []
for fam in KNOWN_FAMILIES:
    for g in MODELS[fam]:
        sel = (S[fam].label == g).values
        cls = raw[fam]["B2"][g]["classes"][sel]
        k_star = T[fam]["B2 (i) max"][f"{g}:k_star"].values[sel]
        others = [m for m in MODELS[fam] if m != g]
        rows.append({"family": fam, "generated by": g,
                     "own model: best class = true class": np.mean(cls[np.arange(sel.sum()), k_star] == S[fam]["class"].values[sel]),
                     "own model: best class = classifier top-1": np.mean(k_star == 0),
                     "own model: mean max weight (ii)": T[fam]["B2 (ii) mix"][f"{g}:w_max"].values[sel].mean(),
                     "other models: mean max weight (ii)": np.mean([T[fam]["B2 (ii) mix"][f"{m}:w_max"].values[sel].mean()
                                                                    for m in others])})
display(pd.DataFrame(rows).set_index(["family", "generated by"]).style.format("{:.3f}")
        .set_caption("B2: classes chosen by the scoring models (all generated images)"))

In [ ]:
def raw_argmax_acc(df, fam):
    # Diagnostic: holdout accuracy of argmax_m s_m (nothing fitted).
    ho = df[df.split == "holdout"]
    s = ho[[f"{m}:sum_logp_guided" for m in MODELS[fam]]].values
    return np.mean(np.array(MODELS[fam])[s.argmax(1)] == ho.label.values)


def fit_eval(df, fam):
    # Section 2's two classifiers, fit on this table's generated train and evaluated on its holdout.
    sizes = MODELS[fam]
    tr, ho = df[df.split == "train"], df[df.split == "holdout"]
    rule = OwnSampleRule(sizes).fit(tr)
    feats = lr_features(fam)
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(tr[feats], tr.label)
    lr_scores = lambda d: pd.DataFrame(lr.predict_log_proba(d[feats]), columns=lr.classes_, index=d.index)
    return {"rule": {**evaluate(rule.scores(ho), ho.label.values, sizes),
                     "train_acc": (np.array(sizes)[rule.scores(tr).values.argmax(1)] == tr.label.values).mean()},
            "lr": {**evaluate(lr_scores(ho), ho.label.values, sizes), "train_acc": lr.score(tr[feats], tr.label)},
            "rule_model": rule, "lr_model": lr}


# Phase A reuses the Section 2 fits (same data, same classifiers); B1 and B2 are refit on their own train.
R = {fam: {ph: {**(oracle[fam] if ph == "A" else fit_eval(T[fam][ph], fam)),
                "raw_argmax_acc": raw_argmax_acc(T[fam][ph], fam)} for ph in PHASES} for fam in KNOWN_FAMILIES}

### A vs B1 vs B2: holdout accuracy

Each row is one classifier, each column group one phase. The change columns separate the two losses: A → B1 is the cost of token recovery, B1 → B2 the cost of the estimated class. Raw $\arg\max_m s_m$ (no fitting, Section 2 diagnostic) is shown for reference.

In [ ]:
NAMES = {"rule": "(a) own-sample rule", "lr": "(b) logistic regression", "raw": "diagnostic: raw argmax s_m"}
rows = []
for fam in KNOWN_FAMILIES:
    for clf, name in NAMES.items():
        row = {"family": fam, "classifier": name}
        for ph in PHASES:
            r = R[fam][ph]
            if clf == "raw":
                row[(ph, "acc")], row[(ph, "95% CI")] = r["raw_argmax_acc"], ""
            else:
                row[(ph, "acc")], row[(ph, "95% CI")] = r[clf]["acc"], f"[{r[clf]['acc_lo']:.3f}, {r[clf]['acc_hi']:.3f}]"
        row[("change", "A -> B1 (tokens)")] = row[("B1", "acc")] - row[("A", "acc")]
        row[("change", "B1 -> B2 (i)")] = row[("B2 (i) max", "acc")] - row[("B1", "acc")]
        row[("change", "B1 -> B2 (ii)")] = row[("B2 (ii) mix", "acc")] - row[("B1", "acc")]
        rows.append(row)
acc3 = pd.DataFrame(rows).set_index(["family", "classifier"])
acc3.columns = pd.MultiIndex.from_tuples(acc3.columns)
display(acc3.style.format("{:.3f}", subset=[c for c in acc3.columns if c[1] == "acc"])
        .format("{:+.3f}", subset=[c for c in acc3.columns if c[0] == "change"]))

In [ ]:
# Holdout confusion matrices for B1 and B2 (A is in Section 2); rows = true size, 512 each.
for fam in KNOWN_FAMILIES:
    for clf in ("rule", "lr"):
        display(pd.concat({ph: R[fam][ph][clf]["cm"] for ph in PHASES[1:]}, axis=1)
                .style.set_caption(f"{fam}, {NAMES[clf]}: holdout confusion"))

### A vs B1 vs B2: pairwise AUC

Holdout AUC per pair of sizes; adjacent pairs yellow, the extreme pair blue. 95% bootstrap intervals for the B2 columns, which the criterion uses.

In [ ]:
for fam in KNOWN_FAMILIES:
    for clf in ("rule", "lr"):
        parts = {}
        for ph in PHASES:
            p = R[fam][ph][clf]["pairs"].set_index(["pair", "type"])
            cols = {"AUC": p.AUC}
            if ph.startswith("B2"):
                cols["CI"] = [f"[{a:.3f}, {b:.3f}]" for a, b in zip(p.lo, p.hi)]
            parts[ph] = pd.DataFrame(cols, index=p.index)
        t = pd.concat(parts, axis=1).reset_index()
        t.columns = [" ".join(c).strip() if isinstance(c, tuple) else c for c in t.columns]
        display(t.style.apply(shade, axis=1).format({c: "{:.3f}" for c in t.columns if c.endswith("AUC")})
                .set_caption(f"{fam}, {NAMES[clf]}: holdout AUC per pair of sizes").hide(axis="index"))

### Margins after recovery

As in the Section 2 diagnostic: the generating model's $s_m$ minus the best other model's, on the holdout, per phase. The margin shrinks as tokens are replaced and the class is estimated; once it goes negative on an image, raw $\arg\max$ gets that image wrong.

In [ ]:
rows = []
for fam in KNOWN_FAMILIES:
    sizes = MODELS[fam]
    for ph in PHASES:
        df = T[fam][ph]
        ho = df[df.split == "holdout"]
        s = ho[[f"{m}:sum_logp_guided" for m in sizes]].values
        own = np.array([sizes.index(l) for l in ho.label])
        other = s.copy(); other[np.arange(len(s)), own] = -np.inf
        margin = s[np.arange(len(s)), own] - other.max(1)
        rows.append({"family": fam, "phase": ph, "share > 0": np.mean(margin > 0), "p1": np.percentile(margin, 1),
                     "p10": np.percentile(margin, 10), "median": np.median(margin),
                     "median per token": np.median(margin) / SEQ_LEN[fam]})
display(pd.DataFrame(rows).set_index(["family", "phase"]).style.format("{:.1f}", subset=["p1", "p10", "median"])
        .format("{:.3f}", subset=["share > 0", "median per token"]))

### Realistic criterion (B2)

Pre-registered: adjacent-size AUC $\ge$ 0.60 on the generated holdout with recovered tokens and estimated classes; a classifier passes if all three adjacent pairs do. The B2 holdout accuracy is shown against the 35% gate for phase C.

In [ ]:
rows = []
for fam in KNOWN_FAMILIES:
    for ph in PHASES[2:]:
        for clf in ("rule", "lr"):
            r = R[fam][ph][clf]
            adj = r["pairs"].query("type == 'adjacent'")
            rows.append({"family": fam, "combination": ph, "classifier": NAMES[clf],
                         "adjacent AUCs": ", ".join(f"{a:.3f}" for a in adj.AUC), "min": adj.AUC.min(),
                         "min CI low": adj.lo.min(), "pass (all adjacent >= 0.60)": bool((adj.AUC >= 0.60).all()),
                         "holdout acc": r["acc"], "phase C gate (acc >= 0.35)": bool(r["acc"] >= 0.35)})
crit3 = pd.DataFrame(rows).set_index(["family", "combination", "classifier"])
display(crit3.style.format({"min": "{:.3f}", "min CI low": "{:.3f}", "holdout acc": "{:.3f}"}))

### Section 3 result

**The realistic criterion passes for both families**, with both classifiers and both class combinations. Holdout 4-way accuracy:

| family | classifier | A (true tokens, true class) | B1 (recovered, true class) | B2 (i) max | B2 (ii) mix | min adjacent AUC, B2 (i) / (ii) |
|---|---|---|---|---|---|---|
| RAR | (a) rule | 98.6% | 97.1% | 96.2% | 96.1% | 0.995 / 0.995 |
| RAR | (b) LR | 100% | 99.9% | 99.5% | 99.5% | 1.000 / 1.000 |
| VAR | (a) rule | 81.8% | 54.3% | 52.4% | 53.0% | 0.698 / 0.699 |
| VAR | (b) LR | 100% | 98.4% | 95.9% | 96.5% | 0.990 / 0.991 |

Every B2 accuracy is above the 35% gate, so phase C (transfer) will be evaluated.

1. **Token recovery is the main loss, and it hits VAR's likelihood hard.** VAR's median own-vs-best-other margin falls from 0.58 nats per token (A) to 0.05 (B1). Only 68% of holdout images still give the generating model the highest $s_m$, against 99.6% for RAR, whose margin only falls from 2.4 to 1.8 per token. The raw $\arg\max$ and the rule lose 28–32 points on VAR at this step. The logistic regression loses only 1.6, so the four models' summaries together still separate the sizes after recovery, even where the single total likelihood $s_m$ no longer does.
2. **The estimated class costs little.** B1 → B2 costs 0.4–0.9 points for RAR and 1.3–2.8 for VAR. The classifier's top-5 contains the true class for 98–99% of images. The generating model's best class within the top 5 is the true class for 98–99% of RAR images and 87–99% of VAR images, and more often for the deeper VAR models.
3. **(i) max and (ii) mixture are practically the same.** The class weights in (ii) are almost one-hot (mean largest weight 0.99–1.00), because the class-conditional likelihoods differ by many nats, so $\log \sum_k \tilde\pi_k e^{s_{mk}} \approx \max_k s_{mk}$. (ii) is 0.6 points higher for VAR with LR, within the intervals.
4. **The rule (a) is not the right classifier here.** It was already the weaker one in phase A, for the reason given in Section 2, and is far below LR for VAR in B1 and B2.

**Caveats.** These are our own generated images, sampled with each repo's ImageNet FID settings (RAR per size, VAR cfg 1.5, top-$k$ 900, top-$p$ 0.96), and the LR is fit to them. Whether the task images were sampled the same way is unknown; phase C (transfer to the task's labelled images) tests exactly that. Which of the LR's features carry the VAR result after recovery has not been examined.

## 4. Transfer to task images (phase C)

**Question.** The classifiers were trained on our own generated images, sampled with each repo's ImageNet FID settings. The task's images were generated with unknown settings. If those differ (RAR guidance $w$, its schedule or temperature; VAR's cfg, top-$k$ or top-$p$), each model's likelihood of its own images shifts, and a classifier fit to our settings can fail on the task.

**Setup.**
- Task images: the in-distribution **train** (100 per size) and **val** (50 per size) images of each family. The **true family is given**, so this measures the size step alone; outliers are not used.
- Scores: `stage4/scores/<family>/task_recovered_topk`. Recovered tokens (Stage 2 $D^{-1}$) are scored with each of the top-5 classes from `stage4/classes/task.npz`, exactly as in B2.
- Class combination: for each family and classifier, the B2 variant ((i) max or (ii) mixture) with the higher **generated-holdout** accuracy in Section 3. Ties are broken by mean adjacent AUC on the holdout, then in favour of (ii). Task images play no part in this choice.
- **Main result:** the Section 3 B2 classifiers (rule (a) and LR (b), trained on generated train only) applied **unchanged** to task train, val and train+val. Accuracy with 95% bootstrap interval, confusion matrix and pairwise AUCs, next to the generated-holdout accuracy. Nothing is refit on task images.

**Pre-registered transfer criterion:** evaluated only if the classifier's B2 holdout accuracy is $\ge$ 35%. If so, it passes when the task accuracy on train+val (150 per size) is within 10 points of the holdout accuracy, $|\text{gap}| \le 10$ points. Below the gate, the task accuracy is still reported, but the criterion is marked "not evaluated".

**Separate comparison (not pre-registered).** The same classifiers and combination, fit on task **train** (100 per size) and evaluated on task **val**. This checks whether the task images hold a size signal our generated data misses. With 50 val images per size, the intervals are about ±7 points.

**Diagnostic: do our sampling settings match the task's?** Per family, the true size × scoring model table of mean `logp_guided` on task train+val, next to the same table on the generated holdout (B2). If the task used our settings, the two tables should look alike, and each size's images should fit its own model's own-sample distribution: $z_{\text{own}} \approx \mathcal N(0, 1)$ under the Section 3 rule.

In [ ]:
# Class combination for the task: per family and classifier, the B2 variant that did best on the GENERATED HOLDOUT.
B2 = ["B2 (i) max", "B2 (ii) mix"]
rows, CHOICE = [], {}
for fam in KNOWN_FAMILIES:
    for clf in ("rule", "lr"):
        cand = []
        for ph in B2:
            r = R[fam][ph][clf]
            cand.append((r["acc"], r["pairs"].query("type == 'adjacent'").AUC.mean(), ph == "B2 (ii) mix", ph))
            rows.append({"family": fam, "classifier": NAMES[clf], "variant": ph, "holdout acc": r["acc"],
                         "holdout correct": int(round(r["acc"] * 2048)), "mean adjacent AUC": cand[-1][1]})
        CHOICE[(fam, clf)] = max(cand)[3]
sel = pd.DataFrame(rows).set_index(["family", "classifier", "variant"])
sel["chosen"] = [CHOICE[(f, {v: k for k, v in NAMES.items()}[c])] == ph for f, c, ph in sel.index]
display(sel.style.format({"holdout acc": "{:.4f}", "mean adjacent AUC": "{:.4f}"})
        .set_caption("choice of class combination, on the generated holdout only"))
print({f"{f} {NAMES[c]}": v for (f, c), v in CHOICE.items()})

In [ ]:
# Task score files: provenance checks, then the in-distribution images of each family, combined both ways.
with np.load(S4 / "classes" / "task.npz") as z:
    CLS_T = {k: z[k] for k in z.files}
key_row = pd.Series(np.arange(len(CLS_T["key"])), index=CLS_T["key"])


def load_task(fam):
    with np.load(S3 / "tokens" / fam / "task.npz") as z:
        tok_key = z["key"]
    raw_t, ids = {}, None
    for m in MODELS[fam]:
        cfg = json.loads((SCORES / fam / "task_recovered_topk" / m / "config.json").read_text())
        assert (cfg["family"], cfg["source"], cfg["tokens"], cfg["class_mode"], cfg["topk"], cfg["model"], cfg["limit"],
                cfg["classes"], sorted(cfg["splits"])) == (fam, "task", "recovered", "topk", 5, m, None, None,
                                                           ["train", "val"]), cfg
        z = np.load(SCORES / fam / "task_recovered_topk" / f"{m}.npz")
        cur = pd.DataFrame({k: z[k] for k in ("row", "key", "split", "label", "family")})
        assert (tok_key[cur.row.values] == cur.key.values).all(), "rows do not match the stage3 token file"
        ids = cur if ids is None else ids
        assert (cur.key.values == ids.key.values).all()
        cl = key_row[cur.key.values].values
        assert (z["classes"] == CLS_T["top10_class"][cl, :5]).all() and np.allclose(z["class_prob"], CLS_T["top10_prob"][cl, :5])
        d = {k: z[k].astype(np.float32) for k in SUMMARIES[fam]}
        assert all(np.isfinite(v).all() for v in d.values()) and (d["sum_logp_guided"] <= 0).all()
        assert np.abs(d["mean_logp_uncond"] - d["mean_logp_uncond"][:, :1]).max() < 1e-3
        d["class_prob"] = z["class_prob"]
        raw_t[m] = d
    keep = (ids.family == fam).values  # in-distribution images of this family (true family given)
    base = ids[keep].reset_index(drop=True)
    out = {}
    for ph, how in [("B2 (i) max", "max"), ("B2 (ii) mix", "mix")]:
        sub = {m: {k: v[keep] for k, v in d.items()} for m, d in raw_t.items()}
        out[ph] = pd.concat([base, combine(sub, fam, how)], axis=1)
    print(fam, "task images:", base.groupby(["label", "split"]).size().unstack().to_dict())
    return out


TT = {fam: load_task(fam) for fam in KNOWN_FAMILIES}

In [ ]:
def clf_scores(fam, ph, clf, df):
    # Class scores of a Section 3 classifier (fit on generated train, phase ph) for the rows of df.
    if clf == "rule":
        return R[fam][ph]["rule_model"].scores(df)
    lr, feats = R[fam][ph]["lr_model"], lr_features(fam)
    return pd.DataFrame(lr.predict_log_proba(df[feats]), columns=lr.classes_, index=df.index)


PARTS = {"task train": ["train"], "task val": ["val"], "task train+val": ["train", "val"]}
C, rows = {}, []
for fam in KNOWN_FAMILIES:
    for clf in ("rule", "lr"):
        ph = CHOICE[(fam, clf)]
        hold = R[fam][ph][clf]
        row = {"family": fam, "classifier": NAMES[clf], "variant": ph, "gen holdout acc": hold["acc"],
               "gen holdout CI": f"[{hold['acc_lo']:.3f}, {hold['acc_hi']:.3f}]"}
        for part, splits in PARTS.items():
            df = TT[fam][ph][TT[fam][ph].split.isin(splits)]
            r = evaluate(clf_scores(fam, ph, clf, df), df.label.values, MODELS[fam])
            row[f"{part} acc"], row[f"{part} CI"] = r["acc"], f"[{r['acc_lo']:.3f}, {r['acc_hi']:.3f}]"
            C[(fam, clf, part)] = r
        tv = C[(fam, clf, "task train+val")]["pairs"]
        row["adjacent AUCs (train+val)"] = ", ".join(f"{a:.3f}" for a in tv.query("type == 'adjacent'").AUC)
        row["extreme AUC (train+val)"] = tv.query("type == 'extreme'").AUC.iloc[0]
        row["gap (holdout - task train+val)"] = row["gen holdout acc"] - row["task train+val acc"]
        rows.append(row)
transfer = pd.DataFrame(rows).set_index(["family", "classifier"])
display(transfer.style.format({c: "{:.3f}" for c in transfer.columns if c.endswith("acc") or c.startswith(("gap", "extreme"))}))

for fam in KNOWN_FAMILIES:
    display(pd.concat({f"{fam} {NAMES[clf]}": C[(fam, clf, "task train+val")]["cm"] for clf in ("rule", "lr")}, axis=1)
            .style.set_caption(f"{fam}: task train+val confusion, generated-trained classifiers (rows = true, 150 per size)"))

In [ ]:
# Pairwise AUC on task train+val, next to the generated holdout (same classifier and variant).
for fam in KNOWN_FAMILIES:
    parts = {}
    for clf in ("rule", "lr"):
        ph = CHOICE[(fam, clf)]
        h = R[fam][ph][clf]["pairs"].set_index(["pair", "type"])
        t = C[(fam, clf, "task train+val")]["pairs"].set_index(["pair", "type"])
        parts[f"{clf}"] = pd.DataFrame({"holdout AUC": h.AUC, "task AUC": t.AUC,
                                        "task CI": [f"[{a:.3f}, {b:.3f}]" for a, b in zip(t.lo, t.hi)]})
    tab = pd.concat(parts, axis=1).reset_index()
    tab.columns = [" ".join(c).strip() if isinstance(c, tuple) else c for c in tab.columns]
    display(tab.style.apply(shade, axis=1).format({c: "{:.3f}" for c in tab.columns if c.endswith("AUC")})
            .set_caption(f"{fam}: pairwise AUC, generated holdout vs task train+val (rule = (a), lr = (b))")
            .hide(axis="index"))

### Transfer criterion

Gate: the classifier's B2 holdout accuracy must be $\ge$ 35% for the criterion to be evaluated. Criterion: $|\text{holdout} - \text{task train+val}| \le 10$ points.

In [ ]:
crit4 = transfer[["variant", "gen holdout acc", "task train+val acc", "task train+val CI", "gap (holdout - task train+val)"]].copy()
crit4["gate (holdout >= 0.35)"] = crit4["gen holdout acc"] >= 0.35
crit4["criterion"] = np.where(crit4["gate (holdout >= 0.35)"],
                              np.where(crit4["gap (holdout - task train+val)"].abs() <= 0.10, "pass", "fail"),
                              "not evaluated (B2 < 35%)")
display(crit4.style.format({"gen holdout acc": "{:.3f}", "task train+val acc": "{:.3f}",
                            "gap (holdout - task train+val)": "{:+.3f}"}))

### Separate comparison: trained on task train, evaluated on task val

Not part of the pre-registered evaluation. Same combination (chosen on the generated holdout) and same classifiers, but the rule's own-sample statistics and the LR are fit on the task's train images only (100 per size), then evaluated on task val (50 per size).

In [ ]:
rows, cms_tt = [], {}
for fam in KNOWN_FAMILIES:
    sizes = MODELS[fam]
    for clf in ("rule", "lr"):
        ph = CHOICE[(fam, clf)]
        df = TT[fam][ph]
        tr, va = df[df.split == "train"], df[df.split == "val"]
        if clf == "rule":
            model = OwnSampleRule(sizes).fit(tr)
            sc_tr, sc_va = model.scores(tr), model.scores(va)
        else:
            feats = lr_features(fam)
            model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(tr[feats], tr.label)
            sc_tr = pd.DataFrame(model.predict_log_proba(tr[feats]), columns=model.classes_, index=tr.index)
            sc_va = pd.DataFrame(model.predict_log_proba(va[feats]), columns=model.classes_, index=va.index)
        r = evaluate(sc_va, va.label.values, sizes)
        cms_tt[(fam, clf)] = r["cm"]
        rows.append({"family": fam, "classifier": NAMES[clf], "variant": ph,
                     "task train acc (in-sample)": np.mean(np.array(sizes)[sc_tr[sizes].values.argmax(1)] == tr.label.values),
                     "task val acc": r["acc"], "task val CI": f"[{r['acc_lo']:.3f}, {r['acc_hi']:.3f}]",
                     "adjacent AUCs (val)": ", ".join(f"{a:.3f}" for a in r["pairs"].query("type == 'adjacent'").AUC),
                     "generated-trained, task val acc": transfer.loc[(fam, NAMES[clf]), "task val acc"]})
task_trained = pd.DataFrame(rows).set_index(["family", "classifier"])
display(task_trained.style.format({c: "{:.3f}" for c in task_trained.columns if "acc" in c})
        .set_caption("COMPARISON ONLY: classifiers fit on task train, evaluated on task val (50 per size)"))
for fam in KNOWN_FAMILIES:
    display(pd.concat({f"{fam} {NAMES[clf]}": cms_tt[(fam, clf)] for clf in ("rule", "lr")}, axis=1)
            .style.set_caption(f"COMPARISON ONLY, {fam}: task val confusion, task-trained classifiers"))

### Diagnostic: sampling settings

Per family, mean `logp_guided` per token (true size × scoring model) on the generated holdout and on task train+val, both B2 with the LR's class combination. Each row is highlighted at its best cell. Below the tables:
- **Row-centred correlation:** each row minus its mean, then the correlation between the generated and task matrices. Row-centring removes the overall difficulty of a size's images and keeps the pattern across scoring models.
- **Own-model fit:** $z_{\text{own}} = (s_{\text{own}} - \mu_{\text{own}})/\sigma_{\text{own}}$, with $\mu, \sigma$ of the B2 rule (the generating model's own samples in generated train). If the task used our settings, $z_{\text{own}}$ on task images is about $\mathcal N(0, 1)$, as on the generated holdout.
- **Raw argmax share:** how often the true size's model gives the highest $s_m$.

In [ ]:
diag_rows = []
for fam in KNOWN_FAMILIES:
    sizes, ph = MODELS[fam], CHOICE[(fam, "lr")]
    gen_ho = T[fam][ph][T[fam][ph].split == "holdout"]
    task = TT[fam][ph]
    tabs = {}
    for name, df in [("generated holdout", gen_ho), ("task train+val", task)]:
        tabs[name] = pd.DataFrame({m: df.groupby("label")[f"{m}:mean_logp_guided"].mean() for m in sizes}).loc[sizes]
    both = pd.concat(tabs, axis=1)
    both.index.name, both.columns.names = "true size", [None, "scored by"]
    display(both.style.format("{:.3f}")
            .highlight_max(axis=1, subset=["generated holdout"], color="#fff3c4")
            .highlight_max(axis=1, subset=["task train+val"], color="#fff3c4")
            .set_caption(f"{fam} ({ph}): mean logp_guided per token"))
    g, t = (tabs[k].sub(tabs[k].mean(axis=1), axis=0).values.ravel() for k in tabs)
    rule = R[fam][CHOICE[(fam, "rule")]]["rule_model"]
    for name, df in [("generated holdout", T[fam][CHOICE[(fam, "rule")]].query("split == 'holdout'")),
                     ("task train+val", TT[fam][CHOICE[(fam, "rule")]])]:
        z = rule.z(df)
        s = df[[f"{m}:sum_logp_guided" for m in sizes]].values
        for size in sizes:
            sel = (df.label == size).values
            diag_rows.append({"family": fam, "true size": size, "data": name,
                              "z_own mean": z.loc[sel, size].mean(), "z_own sd": z.loc[sel, size].std(),
                              "raw argmax = true size": np.mean(s[sel].argmax(1) == sizes.index(size)),
                              "diagonal is row max": bool(tabs[name].loc[size].idxmax() == size)})
    print(f"{fam}: row-centred correlation, generated holdout vs task train+val: {np.corrcoef(g, t)[0, 1]:.3f}")
diag = pd.DataFrame(diag_rows).set_index(["family", "true size", "data"]).unstack("data")
display(diag.style.format("{:.3f}", subset=[c for c in diag.columns if c[0] != "diagonal is row max"])
        .set_caption("own-model fit per size: generated holdout vs task train+val"))

### Section 4 result

**The transfer criterion passes for both families and both classifiers.** All four pass the 35% gate, and the classifiers trained only on our generated images do about as well on the task's labelled images as on the generated holdout:

| family | classifier (combination) | generated holdout | task train + val (150 per size) | gap | criterion |
|---|---|---|---|---|---|
| RAR | (a) rule ((i) max) | 96.2% | 94.3% [92.3, 96.2] | +1.9 | pass |
| RAR | (b) LR ((ii) mix) | 99.5% | 98.7% [97.7, 99.5] | +0.8 | pass |
| VAR | (a) rule ((ii) mix) | 53.0% | 55.5% [51.5, 59.5] | −2.5 | pass |
| VAR | (b) LR ((ii) mix) | 96.5% | 95.5% [93.7, 97.0] | +1.0 | pass |

Task adjacent-pair AUCs are 0.985–1.000 for RAR (both classifiers), 0.995–0.999 for VAR with LR and 0.72–0.81 for VAR with the rule, each within a few hundredths of its holdout value. The combinations were chosen on the generated holdout: (i) max for the RAR rule, (ii) mixture otherwise. The two differ by at most 12 of 2,048 holdout images.

**What the gap implies.**
1. **Our generation settings match the task's.** This time the criterion is informative: the holdout accuracies are far above chance, so a mismatch in sampling settings would have shown up as a drop. The diagnostic agrees:
   - The size × scoring-model tables of mean `logp_guided` on task images reproduce the generated-holdout tables almost exactly (row-centred correlation 1.000 for RAR, 0.999 for VAR). Every cell agrees to within 0.14 nats per token (VAR: within 0.08).
   - On task images, each size fits its own model's own-sample distribution: $z_{\text{own}}$ has mean −0.14 to 0.11 and sd 0.78–1.12 for every size except VAR-d30 (mean 0.21, sd 1.35, n = 150). The holdout values are similar (mean −0.09 to 0.08, sd 0.76–1.14).
   - A different guidance scale, temperature or top-$k$/top-$p$ would shift these likelihoods by far more. The task images were most likely sampled with each repo's ImageNet FID settings, as we did.
2. **The task images hold no size signal our generated data misses.** Fit on task train and evaluated on task val (comparison only), the classifiers reach RAR 90.5% / 99.0% and VAR 58.5% / 95.0% (rule / LR). The generated-trained classifiers on the same val images reach 91.0% / 98.5% and 58.5% / 97.0%; every pair is within its ±7-point interval.
3. **The VAR-d30 row is the one to watch.** Its task images fit their own model slightly better than our holdout does (z mean 0.21; the raw $\arg\max$ picks d30 for 53% of task images against 38% on the holdout). On the task, d30's row has its maximum on the diagonal; on the holdout it does not. With 150 images this may be noise; it does not hurt LR accuracy (147/150 correct).

**Caveat.** The true family is given here. End to end, the size step only applies to images the Stage 2 rule assigns to the right family (96% of val), and outliers are handled before it.

## 5. End to end (phase D)

**Rule.** The Stage 2 decision rule (`NearestFamilyThreshold`, see 002), refit exactly as in 002 from `stage2/metrics.json`: the Stage 2 features, the selected family scores, $q = 0.95$ and the 13 size features. The refit must reproduce 002's val metrics. It gives each image a family (RAR / VAR) or outlier, then a size from a per-family logistic regression on the Stage 2 signals.

**Swap rule (pre-registered).** For each family, the size step is replaced only if one of its Stage 4 classifiers passed **all** of:
- oracle (Section 2): 4-way holdout accuracy $\ge$ 40%;
- realistic B2 (Section 3): every adjacent AUC $\ge$ 0.60, with the class combination used in Section 4;
- transfer (Section 4): evaluated (B2 holdout accuracy $\ge$ 35%) and $|\text{gap}| \le 10$ points.

If both classifiers pass, the one with the higher B2 generated-holdout accuracy is used. For every image the Stage 2 rule assigns to a swapped family, the size becomes that classifier's prediction. The prediction comes from the image's `task_recovered_topk` scores: recovered tokens, top-5 estimated classes, and the same class combination as in Section 4. Family and outlier decisions are unchanged.

**Submission.** Only if val accuracy is higher than Stage 2's. The test images of the swapped families then have to be scored first (`score_likelihood.py --splits test`, into a separate output directory `stage4/scores_test`). The submission is written to `stage4/submission_stage4.csv` once those scores exist.

In [ ]:
# Swap rule: which (family, classifier) passed all three criteria (Sections 2-4)?
rows = []
for fam in KNOWN_FAMILIES:
    for clf in ("rule", "lr"):
        name, ph = NAMES[clf], CHOICE[(fam, clf)]
        c4 = crit4.loc[(fam, name)]
        rows.append({"family": fam, "classifier": name, "combination": ph,
                     "oracle (S2)": bool(crit.loc[(fam, name), "pass"]),
                     "realistic B2 (S3)": bool(crit3.loc[(fam, ph, name), "pass (all adjacent >= 0.60)"]),
                     "transfer (S4)": c4["criterion"] == "pass",
                     "B2 holdout acc": R[fam][ph][clf]["acc"]})
passed = pd.DataFrame(rows).set_index(["family", "classifier"])
passed["all three"] = passed[["oracle (S2)", "realistic B2 (S3)", "transfer (S4)"]].all(axis=1)
display(passed.style.format({"B2 holdout acc": "{:.3f}"}))

SWAP = {}
for fam in KNOWN_FAMILIES:
    ok = [clf for clf in ("rule", "lr") if passed.loc[(fam, NAMES[clf]), "all three"]]
    if ok:
        SWAP[fam] = max(ok, key=lambda clf: R[fam][CHOICE[(fam, clf)]][clf]["acc"])
print("SWAP:", {fam: f"{NAMES[clf]}, {CHOICE[(fam, clf)]}" for fam, clf in SWAP.items()} or "none")

In [ ]:
# Stage 2 rule, refit exactly as in 002 from its saved settings.
S2 = CACHE / "stage2"
m2 = json.loads((S2 / "metrics.json").read_text())
feat = {fam: pd.read_csv(S2 / "features" / f"{fam}.csv") for fam in KNOWN_FAMILIES}
df2 = feat["rar"].merge(feat["var"].drop(columns=["split", "label", "family", "name"]), on="key", validate="1:1")
idx = build_index()
assert df2.key.tolist() == idx.key.tolist()
train2, val2, test2 = (df2[df2.split == s].reset_index(drop=True) for s in ["train", "val", "test"])
rule2 = NearestFamilyThreshold(m2["score_cols"], q=m2["q"], size_cols=m2["size_features"]).fit(train2)
pred2_val = rule2.predict(val2)
stage2_val = report(val2.label, pred2_val)
assert all(np.isclose(stage2_val[k], m2["val"][k]) for k in m2["val"]), "refit does not reproduce Stage 2"
print("Stage 2 rule:", m2["score_cols"], f"q = {m2['q']}", f"| val accuracy {stage2_val['accuracy']:.4f} "
      f"(002: {m2['val']['accuracy']:.4f}), size given family {stage2_val['size_accuracy_given_family']:.4f}")

In [ ]:
# Stage 4 size step for any image: combined top-5 scores of ALL scored task images (every family and outliers),
# indexed by key. Train/val come from stage4/scores, test from stage4/scores_test.
SCORE_ROOTS = {("train", "val"): SCORES, ("test",): S4 / "scores_test"}
_tables = {}


def task_table(fam, splits, how):
    # Combined per-model summaries, one row per scored image of `splits`, indexed by key (cached).
    if (fam, splits, how) in _tables:
        return _tables[(fam, splits, how)]
    root = SCORE_ROOTS[splits] / fam / "task_recovered_topk"
    with np.load(S3 / "tokens" / fam / "task.npz") as z:
        tok_key = z["key"]
    raw_t, keys = {}, None
    for m in MODELS[fam]:
        cfg = json.loads((root / m / "config.json").read_text())
        assert (cfg["family"], cfg["source"], cfg["tokens"], cfg["class_mode"], cfg["topk"], cfg["model"], cfg["limit"],
                cfg["classes"], tuple(sorted(cfg["splits"]))) == (fam, "task", "recovered", "topk", 5, m, None, None,
                                                                  tuple(sorted(splits))), cfg
        z = np.load(root / f"{m}.npz")
        assert (tok_key[z["row"]] == z["key"]).all()
        keys = z["key"] if keys is None else keys
        assert (z["key"] == keys).all()
        cl = key_row[keys].values
        assert (z["classes"] == CLS_T["top10_class"][cl, :5]).all()
        d = {k: z[k].astype(np.float32) for k in SUMMARIES[fam]}
        d["class_prob"] = z["class_prob"]
        raw_t[m] = d
    t = combine(raw_t, fam, how).set_index(pd.Index(keys, name="key"))
    _tables[(fam, splits, how)] = t
    return t


def stage4_size(fam, clf, keys, splits):
    # Size labels from the Stage 4 classifier `clf` of family `fam` (fit on generated train, B2), for these keys.
    ph = CHOICE[(fam, clf)]
    t = task_table(fam, splits, {"B2 (i) max": "max", "B2 (ii) mix": "mix"}[ph]).loc[keys]
    sc = clf_scores(fam, ph, clf, t)[MODELS[fam]]
    return np.array(MODELS[fam])[sc.values.argmax(1)]


# Code check: on the task train/val images of each family, stage4_size reproduces Section 4's predictions.
for fam in KNOWN_FAMILIES:
    for clf in ("rule", "lr"):
        ph = CHOICE[(fam, clf)]
        df = TT[fam][ph]
        ref = np.array(MODELS[fam])[clf_scores(fam, ph, clf, df)[MODELS[fam]].values.argmax(1)]
        assert (stage4_size(fam, clf, df.key.values, ("train", "val")) == ref).all()
print("stage4_size reproduces Section 4 on task train/val")


def predict_stage4(d, swap, splits):
    # Stage 2 labels, with the size replaced for images predicted as a family in `swap`.
    pred = rule2.predict(d).values.copy()
    fam_pred = rule2.predict_family(d).values
    for fam, clf in swap.items():
        m = fam_pred == fam
        if m.any():
            pred[m] = stage4_size(fam, clf, d.key.values[m], splits)
    return pd.Series(pred, index=d.index)


# Forced-swap check: swap both families with their better classifier regardless of the criteria.
forced = {fam: max(("rule", "lr"), key=lambda clf: R[fam][CHOICE[(fam, clf)]][clf]["acc"]) for fam in KNOWN_FAMILIES}
pred_forced = predict_stage4(val2, forced, ("train", "val"))
rep_forced = report(val2.label, pred_forced)
print(f"forced swap {forced}: {(pred_forced.values != pred2_val.values).sum()} of {len(val2)} val labels changed; "
      f"family accuracy {rep_forced['family_accuracy']:.4f} (Stage 2 {stage2_val['family_accuracy']:.4f})")
assert rep_forced["family_accuracy"] == stage2_val["family_accuracy"]
assert (pd.Series(pred_forced).map(lambda l: l if l == "outlier" else l[:3]).values ==
        pd.Series(pred2_val).map(lambda l: l if l == "outlier" else l[:3]).values).all()

In [ ]:
pred4_val = predict_stage4(val2, SWAP, ("train", "val"))
stage4_val = report(val2.label, pred4_val)
print(f"val labels changed by the Stage 4 swap: {(pred4_val.values != pred2_val.values).sum()} of {len(val2)}")
res5 = pd.DataFrame({"Stage 2 val (002)": m2["val"], "Stage 2 val (refit)": stage2_val, "Stage 4 val": stage4_val})
res5["change"] = res5["Stage 4 val"] - res5["Stage 2 val (refit)"]
display(res5.style.format("{:.4f}"))
display(family_confusion(val2.label, pred4_val))
display(confusion(val2.label, pred4_val))

### Test submission

Written only if Stage 4 val accuracy is higher than Stage 2's, and only once the test images have been scored for every swapped family:
```bash
python scripts/score_likelihood.py --family <family> --source task --tokens recovered --class-mode topk \
    --splits test --out /workspace/cache/stage4/scores_test
```
The separate `--out` keeps the train/val scores (`stage4/scores`, a different `--splits` configuration) untouched.

In [ ]:
SUB4 = S4 / "submission_stage4.csv"
if stage4_val["accuracy"] > stage2_val["accuracy"]:
    missing = [fam for fam in SWAP if not all((SCORE_ROOTS[("test",)] / fam / "task_recovered_topk" / f"{m}.npz").exists()
                                              for m in MODELS[fam])]
    if missing:
        print(f"val improved ({stage2_val['accuracy']:.4f} -> {stage4_val['accuracy']:.4f}), but the test scores of "
              f"{missing} are missing. Run, in tmux:")
        for fam in missing:
            print(f"  python scripts/score_likelihood.py --family {fam} --source task --tokens recovered "
                  f"--class-mode topk --splits test --out {S4 / 'scores_test'}")
        print("then rerun this notebook. No submission written.")
    else:
        pred4_test = predict_stage4(test2, SWAP, ("test",))
        sub = pd.DataFrame({"image_name": test2.name, "label": pred4_test.values})
        assert len(sub) == 9800 and sub.image_name.is_unique and sub.image_name.str.endswith(".png").all()
        assert sub.label.isin(LABELS).all()
        sub.to_csv(SUB4, index=False)
        print("val improved: wrote", SUB4)
        sub2 = pd.read_csv(S2 / "submission_stage2.csv")
        assert (sub2.image_name.values == sub.image_name.values).all()
        fam_of = lambda l: l if l == "outlier" else l[:3]
        print(f"test labels changed vs Stage 2: {(sub2.label.values != sub.label.values).mean():.1%}; "
              f"family decisions changed: {(sub2.label.map(fam_of) != sub.label.map(fam_of)).sum()}")
        display(pd.DataFrame({"test Stage 4": sub.label.value_counts(normalize=True),
                              "val pred Stage 4": pred4_val.value_counts(normalize=True),
                              "val true": val2.label.value_counts(normalize=True)}).reindex(LABELS).fillna(0)
                .style.format("{:.3f}"))
else:
    print(f"val accuracy {stage4_val['accuracy']:.4f} is not above Stage 2's {stage2_val['accuracy']:.4f}: "
          f"no submission written")
    if SUB4.exists():
        print(f"WARNING: {SUB4} exists from an earlier run and does not reflect this result")

### Section 5 result

**Both families passed all three pre-registered criteria, so both size steps are replaced**, each with its logistic regression on the B2 mixture (ii). It has the higher B2 holdout accuracy in both families: RAR 99.5% vs 96.2% for the rule, VAR 96.5% vs 53.0%.

| val (450 images) | Stage 2 | Stage 4 |
|---|---|---|
| accuracy | 31.6% | **94.0%** |
| family accuracy | 96.0% | 96.0% |
| outlier recall / precision | 88.0% / 86.3% | 88.0% / 86.3% |
| size accuracy given family | 25.3% | **97.7%** (379 / 388) |

1. **Checks.**
   - The refit reproduces 002's val metrics exactly.
   - The end-to-end size step reproduces Section 4's predictions on task train/val.
   - The forced swap changes 294 of 450 val labels while family accuracy stays at 96.0%, with every family and outlier decision identical. Here the forced swap equals the pre-registered one, since both families passed.
2. **Where the remaining errors are.** Of the 27 val errors, 18 are family or outlier decisions inherited unchanged from Stage 2: 5 RAR images called VAR, 3 RAR and 4 VAR images called outlier, and 6 outliers given a family. Only 9 are wrong sizes. The family/outlier step is now the bottleneck; the size step no longer is.
3. **Submission: not written yet.** Val improved, so the test images of both families have to be scored first. The commands are printed above; they write to `stage4/scores_test`. Rerun this notebook afterwards to write `stage4/submission_stage4.csv`.

## 6. Findings

### Summary: every phase against its pre-registered criterion

Per family and classifier. B1 has no criterion of its own; it only separates the cost of token recovery from that of the class. The end-to-end row applies to the classifier the swap rule selected; size accuracy given family is over val images of that family whose family was predicted correctly.

In [ ]:
def size_acc_given_family(pred, fam):
    true_fam = val2.family.values
    pred_fam = pd.Series(pred).map(lambda l: l if l == "outlier" else l[:3]).values
    m = (true_fam == fam) & (pred_fam == fam)
    return (np.asarray(pred)[m] == val2.label.values[m]).mean()


status = lambda ok: "pass" if ok else "fail"
rows = []
for fam in KNOWN_FAMILIES:
    for clf in ("rule", "lr"):
        name, ph = NAMES[clf], CHOICE[(fam, clf)]
        a, b1, b2 = R[fam]["A"][clf], R[fam]["B1"][clf], R[fam][ph][clf]
        min_adj = lambda r: r["pairs"].query("type == 'adjacent'").AUC.min()
        c4 = crit4.loc[(fam, name)]
        used = SWAP.get(fam) == clf
        rows += [
            {"family": fam, "classifier": name, "phase": "A oracle", "value": f"holdout acc {a['acc']:.3f}",
             "criterion": "4-way acc >= 0.40", "result": status(a["acc"] >= 0.40)},
            {"family": fam, "classifier": name, "phase": "B1 recovered tokens, true class",
             "value": f"holdout acc {b1['acc']:.3f}, min adjacent AUC {min_adj(b1):.3f}",
             "criterion": "none (diagnostic)", "result": "-"},
            {"family": fam, "classifier": name, "phase": f"B2 realistic ({ph})",
             "value": f"min adjacent AUC {min_adj(b2):.3f}, holdout acc {b2['acc']:.3f}",
             "criterion": "every adjacent AUC >= 0.60", "result": status(min_adj(b2) >= 0.60)},
            {"family": fam, "classifier": name, "phase": "C transfer (task train+val)",
             "value": f"task acc {c4['task train+val acc']:.3f}, gap {100 * c4['gap (holdout - task train+val)']:+.1f} pts",
             "criterion": "if B2 acc >= 0.35: |gap| <= 10 pts", "result": c4["criterion"]},
            {"family": fam, "classifier": name, "phase": "D end to end (val)",
             "value": (f"used; val acc {stage4_val['accuracy']:.3f} (Stage 2 {stage2_val['accuracy']:.3f}); "
                       f"{fam} size acc given family {size_acc_given_family(pred4_val.values, fam):.3f} "
                       f"(Stage 2 {size_acc_given_family(pred2_val.values, fam):.3f})") if used
                      else "not used (the other classifier was selected)",
             "criterion": "passes A, B2 and C; val improves",
             "result": status(stage4_val["accuracy"] > stage2_val["accuracy"]) if used else "-"},
        ]
findings = pd.DataFrame(rows).set_index(["family", "classifier", "phase"])
colour = {"pass": "color: #1a7f37", "fail": "color: #cf222e"}
display(findings.style.map(lambda v: colour.get(v, "color: #6e7781" if str(v).startswith("not") else ""), subset=["result"]))

### Where the signal goes: A → B1 → B2

Generated-holdout accuracy and minimum adjacent AUC per phase (B2 with each classifier's chosen combination), with the median per-token margin of the generating model over the best other model. A → B1 is the cost of token recovery, B1 → B2 the cost of the unknown class.

In [ ]:
rows = []
for fam in KNOWN_FAMILIES:
    sizes = MODELS[fam]
    med = {}
    for ph in PHASES:
        ho = T[fam][ph][T[fam][ph].split == "holdout"]
        s = ho[[f"{m}:sum_logp_guided" for m in sizes]].values
        own = np.array([sizes.index(l) for l in ho.label])
        other = s.copy(); other[np.arange(len(s)), own] = -np.inf
        med[ph] = np.median(s[np.arange(len(s)), own] - other.max(1)) / SEQ_LEN[fam]
    for clf in ("rule", "lr"):
        b2 = CHOICE[(fam, clf)]
        r = {ph: R[fam][ph][clf] for ph in ("A", "B1", b2)}
        adj = {ph: v["pairs"].query("type == 'adjacent'").AUC.min() for ph, v in r.items()}
        rows.append({"family": fam, "classifier": NAMES[clf], "B2 combination": b2,
                     "acc A": r["A"]["acc"], "acc B1": r["B1"]["acc"], "acc B2": r[b2]["acc"],
                     "A -> B1": r["B1"]["acc"] - r["A"]["acc"], "B1 -> B2": r[b2]["acc"] - r["B1"]["acc"],
                     "min adj AUC A": adj["A"], "min adj AUC B1": adj["B1"], "min adj AUC B2": adj[b2],
                     "margin/token A": med["A"], "margin/token B1": med["B1"], "margin/token B2": med[b2]})
signal = pd.DataFrame(rows).set_index(["family", "classifier"])
display(signal.style.format("{:.3f}", subset=[c for c in signal.columns if c != "B2 combination"])
        .format("{:+.3f}", subset=["A -> B1", "B1 -> B2"]))

In [ ]:
# Context for the end-to-end result: family decisions of the Stage 2 rule on val and on the (unlabelled) test set.
fam_of = lambda l: l if l == "outlier" else l[:3]
display(pd.DataFrame({"val true": val2.label.map(fam_of).value_counts(normalize=True),
                      "val predicted": pred2_val.map(fam_of).value_counts(normalize=True),
                      "test predicted": rule2.predict_family(test2).value_counts(normalize=True)})
        .reindex(["rar", "var", "outlier"]).style.format("{:.3f}").set_caption("family shares (Stage 2 rule)"))

### Conclusion

**The generator-likelihood hypothesis is supported, and it turns size attribution from chance into near-perfect.** Every pre-registered criterion passes for both families and both classifiers, and the end-to-end val accuracy rises from 31.6% (Stage 2) to 94.0%. Size accuracy given the correct family goes from 25.3% to 97.7%.

1. **The variants differ in their conditional structure, not in their marginal token statistics.** Stage 3 found no size signal in what tokens an image contains (histograms, entropy, repeats, codebook rarity): 26–30% even with the true tokens. Stage 4 asks how probable each token is given its context and class under each model's own guided sampling distribution. With true tokens, the generating model gives the highest likelihood to every holdout image, by a median 2.4 (RAR) and 0.58 (VAR) nats per token. All sizes of a family write tokens from the same codebook with similar overall statistics, but each transformer's sampler leaves its own sequence of choices, and only its own conditional distributions reproduce them.
2. **RAR is easier than VAR, and the difference compounds.**
   - *Sampling settings.* The RAR sizes are sampled with different guidance scales, schedules and temperatures (B: $w = 16$, pow 2.75, $T = 1.0$; L: 15.5, 2.5, 1.02; XL: 6.9, 1.5, 1.02; XXL: 8.0, 1.2, 1.02). Each RAR size's guided distribution therefore differs in its settings as well as its weights. The VAR depths share one setting (cfg 1.5, top-$k$ 900, top-$p$ 0.96) and differ only in the transformer. This is a likely reason for RAR's four times larger oracle margin, but it was not isolated (it would need RAR scored with shared settings).
   - *Token recovery.* $D^{-1}$ recovers about 85% of RAR tokens and 27% of VAR tokens (Stage 3). RAR keeps most of its margin (2.4 → 1.8 nats per token in B1). VAR's falls to 0.05, and the single likelihood $s_m$ then picks the right depth for only 68% of holdout images.
   - *What still works for VAR.* The logistic regression on all four models' summaries (conditional, unconditional and guided log-likelihoods, ranks, Min-20%, top-$k$/top-$p$ support) recovers 98.4% in B1 and 96.5% in B2. The own-sample rule, which standardises each model separately and ignores this joint view, drops to 53%. For the rule and the raw likelihood, token recovery is the main loss (A → B1, 28–32 points); for the logistic regression, recovery and the unknown class each cost under 2 points (1.6 and 1.9). Across classifiers and combinations, the unknown class costs only 0.4–2.8 points (B1 → B2), because the classifier's top-5 almost always contains the true class.
3. **The task was very likely generated with the repos' ImageNet FID settings.** The size × scoring-model likelihood tables on task images match those on our generated holdout (row-centred correlation 1.000 / 0.999). Each size's task images fit its own model's own-sample distribution. Classifiers trained only on generated data lose at most 2.5 points on the task's labelled images.

**What this rules out.** That size information is absent from the images: it is clearly present in the conditional likelihoods. That the task used materially different sampling settings from ours, at least for its train and val images. That the unknown class is a major obstacle.

**What it does not rule out, or leaves open.**
- *Which features carry VAR after recovery.* The LR succeeds where the single likelihood does not, but its features have not been examined (e.g. by ablation).
- *Settings vs weights for RAR*, as above.
- *The test set.* The Stage 2 rule calls about 40% of test images outliers, against 11% of val (table above). Either the test set holds many more outliers than val, or its family/outlier decisions behave differently. The size step cannot fix family or outlier errors, which are now 18 of the 27 val errors.
- *Other generators.* Outliers from other models are still handled by the Stage 2 rule alone; the likelihood scores have not been used for outlier detection.
- *Cost.* Scoring needs every model of a family per image and class: about 1.1 s per image for RAR and 2.7 s for VAR with $K = 5$ on an A40.